In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import os
from pathlib import Path

# Keep Numba/Matplotlib caches somewhere writable in this local env.
os.environ.setdefault("NUMBA_CACHE_DIR", "/private/tmp/numba_cache")
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib")
os.environ.setdefault("XDG_CACHE_HOME", "/private/tmp/xdg_cache")
Path(os.environ["NUMBA_CACHE_DIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
Path(os.environ["XDG_CACHE_HOME"]).mkdir(parents=True, exist_ok=True)

import scanpy as sc
import scvelo as scv
import numpy as np

N_JOBS = 10

adata = sc.read_h5ad("./data/pancreas/pancreas_inferred_velocity.h5ad")
n_pcs = 50

# Use the matrix YOU trust (usually adata.X or adata.layers["Ms"])
X = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X

# Optional: standardize (recommended if you want PCA to behave)
X = StandardScaler(with_mean=True, with_std=True).fit_transform(X)

# Optional: "stretch PCA" (identity by default; customize if needed)
stretch = np.ones(X.shape[1])   # <- replace with gene-wise stretch if desired
X = X * stretch

# PCA
pca = PCA(n_components=n_pcs, svd_solver="arpack", random_state=0)
X_pca = pca.fit_transform(X)

# --------------------------
# OVERWRITE PCA IN ADATA
# --------------------------
adata.obsm["X_pca"] = X_pca
adata.varm["PCs"] = pca.components_.T
adata.uns["pca"] = {
    "variance": pca.explained_variance_,
    "variance_ratio": pca.explained_variance_ratio_,
}

print("✅ Overwrote PCA:", adata.obsm["X_pca"].shape)

scv.tl.velocity(adata, mode="stochastic", vkey="stochastic_velocity")
scv.tl.velocity_graph(adata, vkey="stochastic_velocity", n_jobs=N_JOBS)

scv.tl.velocity(adata, mode="dynamical", vkey="dynamical_velocity")
scv.tl.velocity_graph(adata, vkey="dynamical_velocity", n_jobs=N_JOBS)

if "stochastic_velocity_pca" not in adata.obsm:
    print("Computing stochastic_velocity_pca...")
    scv.tl.velocity_embedding(adata, basis="pca", vkey="stochastic_velocity")

V_pca_stoch = adata.obsm["stochastic_velocity_pca"]

# Compute ONLY if missing
if "dynamical_velocity_pca" not in adata.obsm:
    print("Computing dynamical_velocity_pca...")
    scv.tl.velocity_embedding(adata, basis="pca", vkey="dynamical_velocity")

In [ ]:
import scvelo as scv

scv.settings.verbosity = 3
scv.settings.presenter_view = True
scv.settings.n_jobs = N_JOBS

# Recover full dynamical parameters
scv.tl.recover_dynamics(adata, n_jobs=N_JOBS)

# Latent time
scv.tl.latent_time(adata)

# Velocity confidence
scv.tl.velocity_confidence(
    adata,
    vkey="dynamical_velocity"
)

In [ ]:
scv.pl.scatter(
    adata,
    basis="pca",
    color="latent_time",
    color_map="gnuplot",
    size=20
)

In [ ]:
# scVelo tests differential kinetics for every category in `groupby`.
# It stores per-cell-type p-values in adata.varm["fit_pvals_kinetics"].
# Note: this function does not expose n_jobs; recover_dynamics above uses N_JOBS=10.
scv.tl.differential_kinetic_test(
    adata,
    groupby="clusters",
    var_names="velocity_genes",
    min_cells=10,
)


In [ ]:
import os
import numpy as np
import pandas as pd

outdir = "./data/pancreas"
os.makedirs(outdir, exist_ok=True)

cluster_labels = list(adata.obs["clusters"].cat.categories)
pvals = adata.varm["fit_pvals_kinetics"]

# scVelo stores this as a structured array in some versions and plain array in others.
if getattr(pvals.dtype, "names", None):
    kinetic_pvals = pd.DataFrame(
        {f"pval_{name}": np.asarray(pvals[name]).ravel() for name in pvals.dtype.names},
        index=adata.var_names,
    )
else:
    kinetic_pvals = pd.DataFrame(pvals, index=adata.var_names)
    if kinetic_pvals.shape[1] == len(cluster_labels):
        kinetic_pvals.columns = [f"pval_{cluster}" for cluster in cluster_labels]
    else:
        kinetic_pvals.columns = [f"pval_group_{i}" for i in range(kinetic_pvals.shape[1])]

gene_prioritization = pd.DataFrame(index=adata.var_names)
gene_prioritization["gene"] = gene_prioritization.index
gene_prioritization["min_pval"] = kinetic_pvals.min(axis=1, skipna=True)
gene_prioritization["scvelo_fit_pval_kinetics"] = adata.var["fit_pval_kinetics"]
gene_prioritization["diff_kinetics_clusters"] = adata.var["fit_diff_kinetics"].fillna("")
gene_prioritization["n_diff_kinetics_clusters"] = (
    gene_prioritization["diff_kinetics_clusters"]
    .astype(str)
    .map(lambda x: 0 if x == "" else len(x.split(",")))
)
gene_prioritization = pd.concat([gene_prioritization, kinetic_pvals], axis=1)
gene_prioritization = gene_prioritization.sort_values(
    ["min_pval", "scvelo_fit_pval_kinetics"],
    ascending=True,
    na_position="last",
)

global_out_path = os.path.join(outdir, "pancreas_scvelo_differential_kinetic_gene_prioritization.csv")
gene_prioritization.to_csv(global_out_path, index=False)

per_cell_type_tables = []
for ct in cluster_labels:
    col = f"pval_{ct}"
    if col not in kinetic_pvals.columns:
        continue

    df_ct = pd.DataFrame({
        "cell_type": ct,
        "gene": adata.var_names,
        "pval_kinetics": kinetic_pvals[col].values,
        "scvelo_fit_pval_kinetics": adata.var["fit_pval_kinetics"].values,
        "diff_kinetics_clusters": adata.var["fit_diff_kinetics"].fillna("").astype(str).values,
    })
    df_ct["is_significant"] = df_ct["pval_kinetics"] < 1e-2
    df_ct["is_in_scvelo_diff_cluster_list"] = df_ct["diff_kinetics_clusters"].map(
        lambda x: ct in x.split(",") if x else False
    )
    df_ct = df_ct.sort_values("pval_kinetics", ascending=True, na_position="last")
    df_ct["cell_type_rank"] = np.arange(1, len(df_ct) + 1)
    per_cell_type_tables.append(df_ct)

per_cell_type_kinetics = pd.concat(per_cell_type_tables, ignore_index=True)
per_cell_type_out_path = os.path.join(outdir, "pancreas_scvelo_differential_kinetics_by_cell_type.csv")
per_cell_type_kinetics.to_csv(per_cell_type_out_path, index=False)

top_n = 50
top_per_cell_type = per_cell_type_kinetics.query("cell_type_rank <= @top_n").copy()
top_out_path = os.path.join(outdir, f"pancreas_scvelo_top{top_n}_differential_kinetics_by_cell_type.csv")
top_per_cell_type.to_csv(top_out_path, index=False)

significant_out_path = os.path.join(outdir, "pancreas_scvelo_significant_differential_kinetics_by_cell_type.csv")
per_cell_type_kinetics.query("is_significant").to_csv(significant_out_path, index=False)

print(f"Saved global ranked genes to {global_out_path}")
print(f"Saved per-cell-type ranked genes to {per_cell_type_out_path}")
print(f"Saved top {top_n} per cell type to {top_out_path}")
print(f"Saved significant p < 1e-2 per cell type to {significant_out_path}")

top_per_cell_type.head(50)
